# Up or On the Rocks?
## A Time Series Analysis of Restaurant Job Recovery Post-Covid-19

**Series:** CEU7072200001 — All Employees, Food Services & Drinking Places (NAICS 722)  
**Period:** January 2016 – March 2026  
**Author:** Jason Staats  
**Updated:** May 2026

In [5]:
# --- Core ---
import requests
import json
import pandas as pd
import numpy as np

# --- Visualization ---
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import acf

# --- Time Series ---
from statsmodels.tsa.seasonal import STL
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression

# --- Plot Styling ---
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.family'] = 'sans-serif'

url = "https://api.bls.gov/publicAPI/v2/timeseries/data/"

payload = {
    "seriesid": ["CEU7072200001"],
    "startyear": "2016",
    "endyear": "2026",
    "registrationkey": API_KEY
}

response = requests.post(url, json=payload)

# Debug first
print("Status:", response.status_code)

if response.status_code != 200:
    print(response.text)
else:
    data = response.json()

    records = []
    for item in data['Results']['series'][0]['data']:
        if item['period'].startswith('M'):
            records.append({
                "year": int(item['year']),
                "month": int(item['period'][1:]),
                "value": float(item['value'])
            })

    df = pd.DataFrame(records).sort_values(['year', 'month'])
df['employed'] =  (df['value'] * 1000).round().astype(int)
df = df.drop(columns='value')
df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))
df = df.set_index('date')
df = df.drop(columns=['year', 'month'])
df.head()

Status: 200


KeyError: 'series'

In [ ]:
# =============================================================================
# Section 2: Full Series
# =============================================================================

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index, y=df['employed'],
    mode='lines',
    name='Employed',
    line=dict(color='steelblue', width=2)
))

fig.update_layout(
    title=dict(text='Food Services & Drinking Places Employment<br>January 2016 – 2025',
               x=0.5, font=dict(size=16)),
    xaxis_title='Date',
    yaxis_title='Employees (Millions)',
    yaxis=dict(tickformat='.1fM', tickvals=list(range(6_500_000, 13_000_000, 500_000)),
               ticktext=[f'{v/1_000_000:.1f}M' for v in range(6_500_000, 13_000_000, 500_000)]),
    hovermode='x unified',
    template='plotly_white'
)

fig.show()

## Full Series

Before diving in, it’s worth clarifying what “food services and drinking places” actually includes. This category covers restaurants, bars, food trucks, and other establishments where food and drinks are prepared and served to customers, whether that’s full-service dining, fast food, or takeout.

Employment in food services and drinking places saw steady growth from about 10.95 million in January 2016 to a peak of roughly 12.31 million in June 2019. Going into 2020, it looked like that upward trend would continue, but the COVID-19 pandemic brought that to an abrupt halt. The result was a loss of about 5.7 million jobs from February to April, nearly 50% of total employment in the industry.

Sharp increases in employment were reported in May and June of 2020, but the industry didn’t fully recover to March 2020 levels until April 2022, more than two years later.

Since then, growth has been more gradual, suggesting the recovery has leveled off and the industry has settled into a more stable pattern.

*Note: This data comes from the Current Employment Statistics (CES) and reflects payroll positions, not individual workers. If someone holds two restaurant jobs, they are counted twice. At the same time, workers paid off payroll are not included. So this is best interpreted as a measure of formal payroll activity in the industry rather than a count of unique people.*

In [ ]:
# =============================================================================
# COVID Interpolation
# =============================================================================

df_raw = df.copy()
df_interp = df.copy()
df_interp.loc['2020-03-01':'2021-06-01', 'employed'] = np.nan
df_interp['employed'] = df_interp['employed'].interpolate(method='time')

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_raw.index, y=df_raw['employed'],
    mode='lines', name='Actual',
    line=dict(color='steelblue', width=2)
))

fig.add_trace(go.Scatter(
    x=df_interp.index, y=df_interp['employed'],
    mode='lines', name='Interpolated',
    line=dict(color='coral', width=2, dash='dash')
))

fig.update_layout(
    title=dict(text='Food Services & Drinking Places Employment<br>Actual vs. Interpolated Series',
               x=0.5, font=dict(size=16)),
    xaxis_title='Date',
    yaxis_title='Employees (Millions)',
    yaxis=dict(tickformat='.1fM', tickvals=list(range(6_500_000, 13_000_000, 500_000)),
               ticktext=[f'{v/1_000_000:.1f}M' for v in range(6_500_000, 13_000_000, 500_000)]),
    hovermode='x unified',
    template='plotly_white'
)

fig.show()

## COVID-19 Interpolation

The pandemic created a clear external shock rather than something driven by the underlying trend or normal seasonality in the data. Leaving those values in place would allow a single, extreme event to distort the model and lead to misleading estimates of both trend and seasonality.

To address this, employment values from March 2020 through June 2021 were replaced using linear interpolation based on time. Rather than trying to model the collapse and recovery directly, this approach effectively bridges the gap between pre- and post-pandemic levels, preserving the overall structure of the series without letting the COVID shock dominate the analysis.

The original series is shown alongside the adjusted version above for transparency, and all subsequent analysis is based on the interpolated data.

In [ ]:
# =============================================================================
# Section 3: Seasonality Analysis
# =============================================================================

df_interp['month'] = df_interp.index.month
df_interp['year'] = df_interp.index.year
month_labels = ['Jan','Feb','Mar','Apr','May','Jun',
                'Jul','Aug','Sep','Oct','Nov','Dec']

# STL fit
stl = STL(df_interp['employed'], period=12)
stl_fit = stl.fit()
df_interp['seasonal'] = stl_fit.seasonal

monthly_avg = df_interp.groupby('month')['employed'].mean().reset_index()

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=('Average Employment by Month',
                    'Year-over-Year Employment Overlay',
                    'STL Isolated Seasonal Component'),
    vertical_spacing=0.12
)

# --- Plot 1: Subseries ---
fig.add_trace(go.Bar(
    x=month_labels,
    y=monthly_avg['employed'],
    marker_color='steelblue',
    name='Monthly Average',
    showlegend=False,
    hovertemplate='%{x}: %{customdata}<extra></extra>',
    customdata=[f'{v/1_000_000:.2f}M' for v in monthly_avg['employed']]
), row=1, col=1)

fig.update_yaxes(
    range=[11_000_000, 12_500_000],
    tickvals=list(range(11_000_000, 12_600_000, 200_000)),
    ticktext=[f'{v/1_000_000:.1f}M' for v in range(11_000_000, 12_600_000, 200_000)],
    row=1, col=1
)

# --- Plot 2: Year-over-Year ---
colors = px.colors.qualitative.Plotly
for i, (year, group) in enumerate(df_interp.groupby('year')):
    group = group.sort_values('month')
    fig.add_trace(go.Scatter(
        x=month_labels[:len(group)],
        y=group['employed'],
        mode='lines+markers',
        name=str(year),
        line=dict(color=colors[i % len(colors)], width=1.5),
        marker=dict(size=4),
        hovertemplate=f'{year} %{{x}}: %{{customdata}}<extra></extra>',
        customdata=[f'{v/1_000_000:.2f}M' for v in group['employed']]
    ), row=2, col=1)

fig.update_yaxes(
    tickvals=list(range(11_000_000, 12_600_000, 200_000)),
    ticktext=[f'{v/1_000_000:.1f}M' for v in range(11_000_000, 12_600_000, 200_000)],
    row=2, col=1
)

# --- Plot 3: STL Seasonal Component ---
fig.add_trace(go.Scatter(
    x=df_interp.index,
    y=df_interp['seasonal'],
    mode='lines',
    name='Seasonal Component',
    showlegend=False,
    line=dict(color='steelblue', width=2),
    hovertemplate='%{x|%b %Y}: %{customdata}<extra></extra>',
    customdata=[f'{v/1_000_000:.3f}M' for v in df_interp['seasonal']]
), row=3, col=1)

fig.add_hline(y=0, line_dash='dash', line_color='gray', line_width=0.8, row=3, col=1)

fig.update_yaxes(
    tickvals=list(range(-500_000, 400_000, 100_000)),
    ticktext=[f'{v/1_000_000:.2f}M' for v in range(-500_000, 400_000, 100_000)],
    row=3, col=1
)

fig.update_layout(
    title=dict(text='Food Services & Drinking Places — Seasonality Analysis',
               x=0.5, font=dict(size=16)),
    height=1000,
    template='plotly_white',
    hovermode='closest',
    showlegend=True,
    legend=dict(
        x=1.02,
        y=0.50,
        xanchor='left',
        yanchor='middle'
    )
)

fig.show()

## Seasonality

Employment in food services follows a pretty clear seasonal pattern. It builds through the spring, peaks in the summer months from June through August, and then tapers off through the fall before hitting its lowest point in January and February.

What stands out is how consistent that pattern is across years. Even with everything that happened during COVID, the overall rhythm doesn’t really change.

The year-over-year overlay shows the bigger picture. There’s steady growth from 2016 through 2019, a noticeable dip during the pandemic period, and then a gradual return to growth through 2023 and 2024.

The STL decomposition reinforces this. The direction of the seasonal pattern stays consistent, but the size of the swings changes from year to year. Some summers see stronger spikes than others, which suggests that while the timing of demand is stable, the intensity of that demand can vary.

In [ ]:
# =============================================================================
# Section 4: ACF
# =============================================================================

acf_values = acf(df_interp['employed'], nlags=24)
lags = list(range(len(acf_values)))

fig = go.Figure()

# Zero line
fig.add_hline(y=0, line_color='black', line_width=0.8)

# ACF stems
for i, (lag, val) in enumerate(zip(lags, acf_values)):
    fig.add_trace(go.Scatter(
        x=[lag, lag],
        y=[0, val],
        mode='lines',
        line=dict(color='steelblue', width=2),
        showlegend=False,
        hoverinfo='skip'
    ))

# ACF points
fig.add_trace(go.Scatter(
    x=lags,
    y=acf_values,
    mode='markers',
    marker=dict(color='steelblue', size=6),
    name='ACF',
    showlegend=False,
    hovertemplate='Lag %{x}: %{y:.3f}<extra></extra>'
))

fig.update_layout(
    title=dict(text='Autocorrelation Function (ACF)<br>Food Services & Drinking Places Employment',
               x=0.5, font=dict(size=16)),
    xaxis_title='Lag (Months)',
    yaxis_title='Autocorrelation',
    width=1075,
    template='plotly_white',
    hovermode='closest'
)

fig.show()

## Autocorrelation

The autocorrelation function shows how similar the series is to itself at different time lags, or how much current employment levels relate to past values.

There’s a steady decline from lag 1 through roughly lag 14, with correlations staying strongly positive. That’s what you’d expect from a series with a persistent trend, where recent values carry a lot of information about what comes next.

Around lag 15, correlations cross zero and turn negative. A negative correlation at that point means that if employment is high today, it tends to have been lower about 15 months ago, and vice versa. That makes intuitive sense given the seasonal pattern. Fifteen months back from a summer peak lands somewhere in the spring of the previous year, which is still on the way up but well below the peak that preceded it. The negative values reflect the seasonal cycle working against the correlation rather than any underlying weakness in the series.

Using 24 lags gives a full two-year window, which is enough to capture both the gradual decay from the trend and the seasonal cycle repeating itself. Overall, the pattern reflects a mix of strong trend persistence and clear seasonality.

In [ ]:
# =============================================================================
# Section 5: Full STL Decomposition
# =============================================================================

trend = stl_fit.trend
seasonal = stl_fit.seasonal
residual = stl_fit.resid

fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=('Observed', 'Trend', 'Seasonal', 'Residual'),
    vertical_spacing=0.08
)

# --- Observed ---
fig.add_trace(go.Scatter(
    x=df_interp.index, y=df_interp['employed'],
    mode='lines', name='Observed',
    line=dict(color='steelblue', width=2),
    showlegend=False,
    hovertemplate='%{x|%b %Y}: %{customdata}<extra></extra>',
    customdata=[f'{v/1_000_000:.2f}M' for v in df_interp['employed']]
), row=1, col=1)

# --- Trend ---
fig.add_trace(go.Scatter(
    x=df_interp.index, y=trend,
    mode='lines', name='Trend',
    line=dict(color='steelblue', width=2),
    showlegend=False,
    hovertemplate='%{x|%b %Y}: %{customdata}<extra></extra>',
    customdata=[f'{v/1_000_000:.2f}M' for v in trend]
), row=2, col=1)

# --- Seasonal ---
fig.add_trace(go.Scatter(
    x=df_interp.index, y=seasonal,
    mode='lines', name='Seasonal',
    line=dict(color='steelblue', width=2),
    showlegend=False,
    hovertemplate='%{x|%b %Y}: %{customdata}<extra></extra>',
    customdata=[f'{v/1_000_000:.3f}M' for v in seasonal]
), row=3, col=1)
fig.add_hline(y=0, line_color='gray', line_width=0.8, line_dash='dash', row=3, col=1)

# --- Residual ---
fig.add_trace(go.Bar(
    x=df_interp.index, y=residual,
    name='Residual',
    marker_color='steelblue',
    showlegend=False,
    hovertemplate='%{x|%b %Y}: %{customdata}<extra></extra>',
    customdata=[f'{v/1_000_000:.3f}M' for v in residual]
), row=4, col=1)
fig.add_hline(y=0, line_color='gray', line_width=0.8, line_dash='dash', row=4, col=1)

fig.update_layout(
    title=dict(text='Food Services & Drinking Places — STL Decomposition',
               x=0.5, font=dict(size=16)),
    height=1000,
    width=1075,
    template='plotly_white',
    hovermode='x unified'
)

fig.show()

## STL Decomposition

STL decomposition separates the series into three components: trend, seasonal, and residual.

The trend component mirrors what we saw in the full series. There’s steady growth through 2019, followed by a smooth decline through the pandemic window due to the interpolation, and then a strong recovery through 2023 that begins to level off into 2025.

The seasonal component isolates the repeating monthly pattern independent of the trend. It reinforces the same structure seen earlier, with employment building into the summer and falling off through the winter.

The residual component captures what’s left after removing trend and seasonality. Most of the values cluster fairly close to zero, but there are periods where the deviations are more pronounced. There may be some mild structure here, but nothing strong enough to clearly suggest an additional cycle beyond the yearly pattern.

The observed panel reflects the interpolated series, not the raw data. The smooth transition through 2020 and 2021 is a result of the interpolation rather than actual recorded employment.

In [ ]:
# =============================================================================
# Section 6: Forecasting
# =============================================================================

from sklearn.linear_model import LinearRegression

df_interp.index = pd.DatetimeIndex(df_interp.index, freq='MS')

# --- Train/Validation Split ---
train = df_interp[df_interp.index < '2025-01-01']['employed']
validation = df_interp[(df_interp.index >= '2025-01-01') & 
                        (df_interp.index < '2026-01-01')]['employed']

print(f"Training observations: {len(train)}")
print(f"Validation observations: {len(validation)}")

In [ ]:
# --- Model 1: ETS (Exponential Smoothing) ---
ets_model = ExponentialSmoothing(
    train,
    trend='add',
    seasonal='add',
    seasonal_periods=12
).fit(optimized=True)
ets_forecast = ets_model.forecast(12)

# --- Model 2: SARIMA ---
sarima_model = SARIMAX(
    train,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12)
).fit(disp=False)
sarima_forecast = sarima_model.forecast(12)

# --- Model 3: Seasonal Naive ---
snaive_forecast = pd.Series(
    train.iloc[-12:].values,
    index=validation.index
)

# --- Model 4: OLS with Trend and Seasonal Dummies ---
def build_ols_features(series):
    df_ols = pd.DataFrame({'employed': series})
    df_ols['trend'] = range(len(df_ols))
    for m in range(1, 12):
        df_ols[f'month_{m}'] = (df_ols.index.month == m).astype(int)
    return df_ols

train_ols = build_ols_features(train)
X_train = train_ols.drop(columns='employed')
y_train = train_ols['employed']

ols_model = LinearRegression().fit(X_train, y_train)

future_ols = pd.DataFrame({'trend': range(len(train), len(train) + 12)},
                           index=validation.index)
for m in range(1, 12):
    future_ols[f'month_{m}'] = (future_ols.index.month == m).astype(int)

ols_forecast = pd.Series(ols_model.predict(future_ols), index=validation.index)

# --- MAE Comparison ---
models = {
    'ETS': ets_forecast,
    'SARIMA': sarima_forecast,
    'Seasonal Naive': snaive_forecast,
    'OLS': ols_forecast
}

print("Model Validation MAE (January - December 2025):")
print("-" * 40)
for name, forecast in models.items():
    mae = mean_absolute_error(validation, forecast)
    print(f"{name:20s}: {mae:,.0f} employees")

In [ ]:
# --- Validation Plot ---
fig = go.Figure()

# Actual training data
fig.add_trace(go.Scatter(
    x=train.index, y=train.values,
    mode='lines', name='Training Data',
    line=dict(color='gray', width=1.5),
    hovertemplate='%{x|%b %Y}: %{customdata}<extra></extra>',
    customdata=[f'{v/1_000_000:.2f}M' for v in train.values]
))

# Actual validation data
fig.add_trace(go.Scatter(
    x=validation.index, y=validation.values,
    mode='lines', name='Actual 2025',
    line=dict(color='black', width=2),
    hovertemplate='%{x|%b %Y}: %{customdata}<extra></extra>',
    customdata=[f'{v/1_000_000:.2f}M' for v in validation.values]
))

# ETS forecast
fig.add_trace(go.Scatter(
    x=validation.index, y=ets_forecast.values,
    mode='lines', name=f'ETS (MAE: {mean_absolute_error(validation, ets_forecast):,.0f})',
    line=dict(color='steelblue', width=2, dash='dash'),
    hovertemplate='%{x|%b %Y}: %{customdata}<extra></extra>',
    customdata=[f'{v/1_000_000:.2f}M' for v in ets_forecast.values]
))

# SARIMA forecast
fig.add_trace(go.Scatter(
    x=validation.index, y=sarima_forecast.values,
    mode='lines', name=f'SARIMA (MAE: {mean_absolute_error(validation, sarima_forecast):,.0f})',
    line=dict(color='coral', width=2, dash='dash'),
    hovertemplate='%{x|%b %Y}: %{customdata}<extra></extra>',
    customdata=[f'{v/1_000_000:.2f}M' for v in sarima_forecast.values]
))

# Seasonal Naive forecast
fig.add_trace(go.Scatter(
    x=validation.index, y=snaive_forecast.values,
    mode='lines', name=f'Seasonal Naive (MAE: {mean_absolute_error(validation, snaive_forecast):,.0f})',
    line=dict(color='mediumseagreen', width=2, dash='dash'),
    hovertemplate='%{x|%b %Y}: %{customdata}<extra></extra>',
    customdata=[f'{v/1_000_000:.2f}M' for v in snaive_forecast.values]
))

fig.update_layout(
    title=dict(text='Food Services & Drinking Places — Model Validation<br>Forecast vs. Actual 2025',
               x=0.5, font=dict(size=16)),
    xaxis_title='Date',
    yaxis_title='Employees (Millions)',
    yaxis=dict(
        tickvals=list(range(11_000_000, 13_000_000, 200_000)),
        ticktext=[f'{v/1_000_000:.1f}M' for v in range(11_000_000, 13_000_000, 200_000)]
    ),
    template='plotly_white',
    hovermode='x unified',
    xaxis=dict(range=['2023-06-01', '2026-01-01'])
)

fig.show()

In [ ]:
# --- Forward Forecast: 2026 ---
# Retrain SARIMA on full series
full_series = df_interp['employed']
full_series.index = pd.DatetimeIndex(full_series.index, freq='MS')

sarima_final = SARIMAX(
    full_series,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12)
).fit(disp=False)

forecast_index = pd.date_range(start='2026-01-01', periods=12, freq='MS')
sarima_2026 = sarima_final.forecast(12)
sarima_2026.index = forecast_index

fig = go.Figure()

# Historical series
fig.add_trace(go.Scatter(
    x=full_series.index, y=full_series.values,
    mode='lines', name='Historical',
    line=dict(color='gray', width=1.5),
    hovertemplate='%{x|%b %Y}: %{customdata}<extra></extra>',
    customdata=[f'{v/1_000_000:.2f}M' for v in full_series.values]
))

# 2026 forecast
fig.add_trace(go.Scatter(
    x=sarima_2026.index, y=sarima_2026.values,
    mode='lines+markers', name='SARIMA 2026 Forecast',
    line=dict(color='coral', width=2, dash='dash'),
    marker=dict(size=6),
    hovertemplate='%{x|%b %Y}: %{customdata}<extra></extra>',
    customdata=[f'{v/1_000_000:.2f}M' for v in sarima_2026.values]
))

# Actual Jan-Mar 2026 for immediate validation
actual_2026 = df_interp[df_interp.index >= '2026-01-01']['employed']
fig.add_trace(go.Scatter(
    x=actual_2026.index, y=actual_2026.values,
    mode='lines+markers', name='Actual 2026 (Jan-Mar)',
    line=dict(color='steelblue', width=2),
    marker=dict(size=6),
    hovertemplate='%{x|%b %Y}: %{customdata}<extra></extra>',
    customdata=[f'{v/1_000_000:.2f}M' for v in actual_2026.values]
))

fig.update_layout(
    title=dict(text='Food Services & Drinking Places — SARIMA 2026 Forecast',
               x=0.5, font=dict(size=16)),
    xaxis_title='Date',
    yaxis_title='Employees (Millions)',
    yaxis=dict(
        tickvals=list(range(11_000_000, 13_500_000, 200_000)),
        ticktext=[f'{v/1_000_000:.1f}M' for v in range(11_000_000, 13_500_000, 200_000)]
    ),
    template='plotly_white',
    hovermode='x unified'
)

fig.show()

## Forecasting

To evaluate how well each model could predict unseen data, the most recent 12 months of the series, January through December 2025, were held back as a validation set, representing one complete seasonal cycle. Four models were 
trained on January 2016 through December 2024 and then asked to forecast the 12 months 
they had never seen. Their predictions were compared against the actual 2025 values using 
Mean Absolute Error, which measures the average size of the forecast miss in raw employee 
counts.

| Model | MAE |
|---|---|
| ETS | 51,719 |
| SARIMA | 45,851 |
| Seasonal Naive | 49,442 |
| OLS | 172,317 |

SARIMA produced the lowest average error, missing by roughly 46,000 employees per month 
against a baseline of approximately 12 million, which is less than half a percent. All 
three competitive models captured the seasonal shape of 2025 accurately, with a modest 
upward bias emerging in the later months of the year, suggesting the sector may be 
settling into a slower growth phase after the post-pandemic rebound. The ETS model 
produced a convergence warning during fitting, meaning its parameters may not be fully 
optimized, though it remained competitive. OLS struggled significantly, as a straight-line 
model lacks the flexibility to handle the curves and seasonality in this data.

SARIMA was retrained on the full series and used to forecast all of 2026. The model 
projects employment staying within roughly the 11.8 to 12.5 million range with the 
expected summer peak. However, actual data from January through March 2026 has come in 
below forecast. This gap suggests the sector may be softening beyond what historical patterns would 
predict. As monthly BLS releases become available throughout 2026, the forecast will 
be evaluated against realized outcomes.